In [3]:
"""
fetch_weather_openmeteo.py
==========================
Pobiera prawdziwe dane pogodowe z Open-Meteo dla Katowic (2023-2024)
w dokładnie tej samej strukturze co solar_dataset_tauron.csv

Uruchomienie:
    pip install requests pandas
    python fetch_weather_openmeteo.py

Wynik:
    weather_katowice_real.csv  — te same kolumny co syntetyczny dataset
                                 (bez production_mwh)
"""

import requests
import pandas as pd
import numpy as np

# konfiguracja
LATITUDE   = 50.29
LONGITUDE  = 19.02
START_DATE = "2023-01-01"
END_DATE   = "2025-12-31"
TIMEZONE   = "Europe/Warsaw"
OUTPUT     = "weather_gliwice_real.csv"

#pobieranie

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude":   LATITUDE,
    "longitude":  LONGITUDE,
    "start_date": START_DATE,
    "end_date":   END_DATE,
    "hourly": [
        "uv_index",               # uv_index         (0–11)
        "cloud_cover",            # cloud_cover_pct   (%)
        "sunshine_duration",      # → sunshine_pct    (sek/h → %)
        "temperature_2m",         # temperature_c     (°C)
        "wind_speed_10m",         # wind_speed_ms     (m/s)
        "relative_humidity_2m",   # humidity_pct      (%)
        "surface_pressure",       # pressure_hpa      (hPa)
    ],
    "timezone": TIMEZONE,
}

response = requests.get(url, params=params, timeout=30)

if response.status_code != 200:
    print(f"Błąd API: {response.status_code}")
    print(response.text)
    exit(1)

data = response.json()
print("Pobrano dane OK")

# ── BUDOWANIE DATAFRAME ───────────────────────────────────────────────────────
hourly = data["hourly"]

df = pd.DataFrame({
    "timestamp":       pd.to_datetime(hourly["time"]),
    "uv_index":        hourly["uv_index"],
    "cloud_cover_pct": hourly["cloud_cover"],
    "sunshine_pct":    hourly["sunshine_duration"],   # tymczasowo w sekundach
    "temperature_c":   hourly["temperature_2m"],
    "wind_speed_ms":   hourly["wind_speed_10m"],
    "humidity_pct":    hourly["relative_humidity_2m"],
    "pressure_hpa":    hourly["surface_pressure"],
})

# ── TRANSFORMACJE — dokładnie jak w syntetycznym datasecie ────────────────────

# sunshine_duration to sekundy słońca w danej godzinie (0–3600)
# zamieniamy na procent godziny (0–100%)
df["sunshine_pct"] = (df["sunshine_pct"] / 3600 * 100).round(1)

# kolumny czasowe
df["date"]        = df["timestamp"].dt.date
df["hour"]        = df["timestamp"].dt.hour
df["day_of_year"] = df["timestamp"].dt.day_of_year
df["month"]       = df["timestamp"].dt.month

# uzupełnij NaN — Open-Meteo czasem zwraca None dla UV w nocy
df["uv_index"]    = df["uv_index"].fillna(0.0)
df["sunshine_pct"]= df["sunshine_pct"].fillna(0.0)

# zaokrąglenia — identyczne jak w syntetycznym datasecie
df["uv_index"]        = df["uv_index"].round(2)
df["cloud_cover_pct"] = df["cloud_cover_pct"].round(1)
df["temperature_c"]   = df["temperature_c"].round(1)
df["wind_speed_ms"]   = df["wind_speed_ms"].round(1)
df["humidity_pct"]    = df["humidity_pct"].round(1)
df["pressure_hpa"]    = df["pressure_hpa"].round(1)



# typy — identyczne jak w syntetycznym datasecie
df["hour"]        = df["hour"].astype("int8")
df["day_of_year"] = df["day_of_year"].astype("int16")
df["month"]       = df["month"].astype("int8")

float_cols = [
    "uv_index", "cloud_cover_pct", "sunshine_pct",
    "temperature_c", "wind_speed_ms", "humidity_pct", "pressure_hpa"
]
df[float_cols] = df[float_cols].astype("float32")

# ── KOLEJNOŚĆ KOLUMN — identyczna jak w solar_dataset_tauron.csv ─────────────
df = df[[
    "timestamp", "date", "hour", "day_of_year", "month",
    "uv_index", "cloud_cover_pct", "sunshine_pct",
    "temperature_c", "wind_speed_ms", "humidity_pct", "pressure_hpa",
    # production_mwh celowo pominięte — dodasz przez PVlib osobno
]]

# ── ZAPIS ─────────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT, index=False)

print(f"\nZapisano: {OUTPUT}")
print(f"Wierszy:  {len(df):,}")
print(f"Zakres:   {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"\nStatystyki:")
print(df[float_cols].describe().round(2))
print(f"\nPierwsze 3 wiersze:")
print(df.head(3).to_string())

Pobrano dane OK

Zapisano: weather_gliwice_real.csv
Wierszy:  26,304
Zakres:   2023-01-01 00:00:00 → 2025-12-31 23:00:00

Statystyki:
       uv_index  cloud_cover_pct  sunshine_pct  temperature_c  wind_speed_ms  \
count   26304.0         26304.00      26304.00       26304.00       26304.00   
mean        0.0            66.70         32.36          10.60          12.04   
std         0.0            40.44         45.48           8.52           6.21   
min         0.0             0.00          0.00         -18.20           0.00   
25%         0.0            23.00          0.00           3.70           7.60   
50%         0.0            96.00          0.00          10.10          10.80   
75%         0.0           100.00        100.00          17.20          15.60   
max         0.0           100.00        100.00          34.90          39.80   

       humidity_pct  pressure_hpa  
count      26304.00      26304.00  
mean          75.01        980.60  
std           17.38          8.29  
m

In [2]:
from api_for_weather.class_weather_api import WeatherAPI

a = WeatherAPI()



In [3]:
a.upload_weather_data()
a.shopweather_api()

            timestamp uv_index  cloud_cover_pct  sunshine_pct  temperature_c  \
0 2023-01-01 00:00:00      0.0              100           0.0           10.4   
1 2023-01-01 01:00:00      0.0              100           0.0           10.4   
2 2023-01-01 02:00:00      0.0               99           0.0           11.0   
3 2023-01-01 03:00:00      0.0              100           0.0           10.6   
4 2023-01-01 04:00:00      0.0               98           0.0           10.9   

   wind_speed_ms  humidity_pct  pressure_hpa        date  hour  day_of_year  \
0           26.6            70         984.1  2023-01-01     0            1   
1           24.8            69         983.9  2023-01-01     1            1   
2           25.5            69         984.0  2023-01-01     2            1   
3           23.2            69         984.1  2023-01-01     3            1   
4           23.2            66         984.0  2023-01-01     4            1   

   month  hour_sin  hour_cos   doy_sin   doy

In [5]:
a.csv_weather_api()
